# LeetCode 2063 — Vowels of All Substrings bằng Python

Link đề: https://leetcode.com/problems/vowels-of-all-substrings/ · Độ khó: **Medium** · Chủ đề: Math, String, Dynamic Programming, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def countVowels(self, word: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi góc nhìn "**tổng theo chuỗi con**" thành "**tổng theo đóng góp của từng ký tự**" (contribution technique).
2. Đếm số chuỗi con chứa vị trí `i`: `(i + 1) · (n − i)` (chọn đầu trong `[0, i]`, cuối trong `[i, n−1]`).
3. Cách DP tương đương: `f[i]` = tổng số nguyên âm trên mọi chuỗi con **kết thúc** tại `i`.
4. Lưu ý độ lớn: đáp án tới `~n³/6 ≈ 1.7·10¹⁴` — vượt `int32`.
5. Kiểm chứng bằng vét cạn `O(n²)`/`O(n³)`.

## 2. Đề bài & ví dụ

Cho chuỗi `word`, trả về **tổng số nguyên âm** (`'a'`, `'e'`, `'i'`, `'o'`, `'u'`) trên **mọi chuỗi con** (liên tiếp, khác rỗng) của `word`. Đáp án có thể không vừa số nguyên 32-bit có dấu.

**Ví dụ 1:** `"aba"` → `6` (chuỗi con `"a"`, `"ab"`, `"ba"`, `"a"` mỗi chuỗi 1 nguyên âm; `"aba"` 2; `"b"` 0).

**Ví dụ 2:** `"abc"` → `3`.

**Ví dụ 3:** `"ltcd"` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; chữ thường.

In [1]:
import random
import string
import time
from typing import List

VOWELS = set("aeiou")
examples = [("aba", 6), ("abc", 3), ("ltcd", 0)]
for w, exp in examples:
    print(f"{w!r} -> kỳ vọng {exp}")

'aba' -> kỳ vọng 6
'abc' -> kỳ vọng 3
'ltcd' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Số chuỗi con

Chuỗi dài `n` có `n(n+1)/2` chuỗi con (chọn cặp đầu ≤ cuối). Với `n = 10⁵` là `5·10⁹` — không thể liệt kê.

### 3.2 Đổi thứ tự lấy tổng

`Σ_{chuỗi con S} (số nguyên âm trong S) = Σ_{S} Σ_{i ∈ S} [word[i] là nguyên âm] = Σ_{i nguyên âm} (số chuỗi con chứa i)`.

In [2]:
w = "aba"; n = len(w)
subs = [(l, r) for l in range(n) for r in range(l, n)]
by_sub = sum(sum(ch in VOWELS for ch in w[l:r + 1]) for l, r in subs)
by_pos = sum(sum(1 for l, r in subs if l <= i <= r) for i in range(n) if w[i] in VOWELS)
print("tổng theo chuỗi con:", by_sub, "| tổng theo vị trí:", by_pos)

tổng theo chuỗi con: 6 | tổng theo vị trí: 6


### 3.3 Số chuỗi con chứa vị trí `i`

Đầu `l ∈ [0, i]` (`i + 1` cách), cuối `r ∈ [i, n−1]` (`n − i` cách) ⇒ `(i + 1)(n − i)` chuỗi con.

In [3]:
n = 5
print([(i, (i + 1) * (n - i), sum(1 for l in range(n) for r in range(l, n) if l <= i <= r)) for i in range(n)])

[(0, 5, 5), (1, 8, 8), (2, 9, 9), (3, 8, 8), (4, 5, 5)]


### 3.4 Độ lớn đáp án

Nếu mọi ký tự là nguyên âm, đáp án `= Σ (i+1)(n−i) = n(n+1)(n+2)/6`; với `n = 10⁵` là khoảng `1.67·10¹⁴`.

In [4]:
n = 10 ** 5
print(f"{n * (n + 1) * (n + 2) // 6:,} > 2^31 - 1 = {2 ** 31 - 1:,}")

166,671,666,700,000 > 2^31 - 1 = 2,147,483,647


### 3.5 Kỹ thuật đóng góp (contribution technique) nói chung

Khi đề hỏi `Σ_{đối tượng lớn} g(đối tượng)` và `g` là **tổng** của các phần nhỏ (`g(S) = Σ_{x ∈ S} h(x)`), ta đổi thứ tự: `Σ_x h(x) · (số đối tượng lớn chứa x)`. Bài toán trở thành **đếm** — thường có công thức đóng. Ví dụ quen thuộc: tổng mọi mảng con của `a` là `Σ a[i]·(i+1)(n−i)`; tổng mọi mảng con độ dài lẻ (LeetCode 1588) dùng số mảng con **lẻ** chứa `i`.

In [5]:
a = [1, 4, 2, 5, 3]; n = len(a)
direct = sum(sum(a[l:r + 1]) for l in range(n) for r in range(l, n))
contrib = sum(a[i] * (i + 1) * (n - i) for i in range(n))
odd_direct = sum(sum(a[l:r + 1]) for l in range(n) for r in range(l, n) if (r - l) % 2 == 0)
odd_contrib = sum(a[i] * (((i + 1) * (n - i) + 1) // 2) for i in range(n))
print("tổng mọi mảng con:", direct, "=", contrib, "| độ dài lẻ (1588):", odd_direct, "=", odd_contrib)

tổng mọi mảng con: 110 = 110 | độ dài lẻ (1588): 58 = 58


## 4. Ý tưởng nền tảng / trực giác

Mỗi nguyên âm ở vị trí `i` được "đếm" một lần trong mỗi chuỗi con chứa nó ⇒ đóng góp `(i + 1)(n − i)`. Đáp án là tổng đóng góp của các nguyên âm.

Tính tay **Ví dụ 1** `"aba"` (`n = 3`):

| i | ký tự | nguyên âm? | (i+1)(n−i) |
|---|---|---|---|
| 0 | a | ✓ | 1·3 = 3 |
| 1 | b | ✗ | — |
| 2 | a | ✓ | 3·1 = 3 |

Tổng `6` ✓.

In [6]:
for w, exp in examples:
    n = len(w); parts = [(i, (i + 1) * (n - i)) for i in range(n) if w[i] in VOWELS]
    print(f"{w}: đóng góp {parts} -> {sum(p for _, p in parts)} (kỳ vọng {exp})")

aba: đóng góp [(0, 3), (2, 3)] -> 6 (kỳ vọng 6)
abc: đóng góp [(0, 3)] -> 3 (kỳ vọng 3)
ltcd: đóng góp [] -> 0 (kỳ vọng 0)


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi con, đếm cộng dồn

Cố định đầu `l`, mở rộng `r` và cộng dồn số nguyên âm — `O(n²)` (không cần `O(n³)`).

In [7]:
def vowels_bruteforce(word):
    n = len(word); total = 0
    for l in range(n):
        cnt = 0
        for r in range(l, n):
            cnt += word[r] in VOWELS
            total += cnt
    return total

for w, exp in examples:
    got = vowels_bruteforce(w); print(got, "kỳ vọng", exp); assert got == exp

6 kỳ vọng 6
3 kỳ vọng 3
0 kỳ vọng 0


**Độ phức tạp:** `O(n²)` = `5·10⁹` với `n = 10⁵` ⇒ TLE.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đóng góp):** đáp án `= Σ_{i: word[i] ∈ VOWELS} (i + 1)(n − i)` — một lượt `O(n)`.

**Insight 2 (DP kết thúc tại `i`):** gọi `f[i]` = tổng số nguyên âm trên các chuỗi con **kết thúc** ở `i`. Mỗi chuỗi con kết thúc ở `i − 1` kéo dài thêm `word[i]`, cộng thêm chuỗi con `"word[i]"` ⇒ `f[i] = f[i−1] + (i + 1)·[word[i] là nguyên âm]` (có `i + 1` chuỗi con kết thúc ở `i`, mỗi chuỗi nhận thêm 1 nếu `word[i]` là nguyên âm). Đáp án `= Σ f[i]`.

**Invariant (DP):** sau bước `i`, `f` là tổng số nguyên âm trên mọi chuỗi con kết thúc ở `i`, `total` là tổng trên mọi chuỗi con kết thúc ở vị trí `≤ i`.

### 6.1 Minh họa: đóng góp theo vị trí có dạng parabol

`(i + 1)(n − i)` lớn nhất ở giữa chuỗi: nguyên âm ở giữa đóng góp nhiều hơn nguyên âm ở mép.

In [8]:
n = 9
print("vị trí  :", " ".join(f"{i:>3}" for i in range(n)))
print("đóng góp:", " ".join(f"{(i + 1) * (n - i):>3}" for i in range(n)))

vị trí  :   0   1   2   3   4   5   6   7   8
đóng góp:   9  16  21  24  25  24  21  16   9


### 6.2 Bảng "chuỗi con × vị trí" cho `"aba"`

Mỗi hàng là một chuỗi con, mỗi cột là một vị trí; ô có dấu `x` nếu chuỗi con chứa vị trí đó. Tổng theo **hàng** (chỉ đếm cột nguyên âm) là cách tính "theo chuỗi con"; tổng theo **cột** là cách tính "theo đóng góp". Hai cách đọc cùng một bảng nên luôn bằng nhau — đây là toàn bộ chứng minh bằng hình ảnh.

In [9]:
w = "aba"; n = len(w)
print("chuỗi con | " + " ".join(f"{i}:{w[i]}" for i in range(n)) + " | số nguyên âm")
col = [0] * n; total = 0
for l in range(n):
    for r in range(l, n):
        row = ["x  " if l <= i <= r else ".  " for i in range(n)]
        cnt = sum(w[i] in VOWELS for i in range(l, r + 1)); total += cnt
        for i in range(l, r + 1): col[i] += 1
        print(f"{w[l:r + 1]:<9} | " + " ".join(row) + f" | {cnt}")
print("số chuỗi con chứa mỗi vị trí:", col, "| tổng theo hàng:", total, "| tổng theo cột (nguyên âm):", sum(col[i] for i in range(n) if w[i] in VOWELS))

chuỗi con | 0:a 1:b 2:a | số nguyên âm
a         | x   .   .   | 1
ab        | x   x   .   | 1
aba       | x   x   x   | 2
b         | .   x   .   | 0
ba        | .   x   x   | 1
a         | .   .   x   | 1
số chuỗi con chứa mỗi vị trí: [3, 4, 3] | tổng theo hàng: 6 | tổng theo cột (nguyên âm): 6


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đóng góp từng vị trí (bản nộp)

In [10]:
class Solution:
    def countVowels(self, word: str) -> int:
        n = len(word)
        return sum((i + 1) * (n - i) for i, ch in enumerate(word) if ch in "aeiou")

print([Solution().countVowels(w) for w, _ in examples])

[6, 3, 0]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — DP theo vị trí kết thúc

In [11]:
def vowels_dp(word):
    f = total = 0
    for i, ch in enumerate(word):
        if ch in VOWELS:
            f += i + 1
        total += f
    return total

print([vowels_dp(w) for w, _ in examples])

[6, 3, 0]


**Độ phức tạp Cách 3:** `O(n)`, `O(1)`.

### 7.3 Cách 4 — Tổng tiền tố của tổng tiền tố

Gọi `P[k]` = số nguyên âm trong `word[0..k−1]`. Số nguyên âm của `word[l..r]` là `P[r+1] − P[l]`. Tổng qua mọi `l ≤ r`: `Σ_{j} P[j]·(số lần xuất hiện với dấu +) − …` ⇒ `Σ_{j=0}^{n} P[j]·(2j − n)`. Công thức này đúng vì mỗi `P[j]` xuất hiện với dấu `+` khi `j = r + 1` (có `j` cách chọn `l < j`) và dấu `−` khi `j = l` (có `n − j` cách chọn `r ≥ j`).

In [12]:
def vowels_prefix(word):
    n = len(word); P = [0] * (n + 1)
    for i, ch in enumerate(word):
        P[i + 1] = P[i] + (ch in VOWELS)
    return sum(P[j] * (2 * j - n) for j in range(n + 1))

print([vowels_prefix(w) for w, _ in examples])

[6, 3, 0]


### 7.4 Cách 5 — Đếm theo độ dài chuỗi con (cửa sổ trượt cho mỗi độ dài)

Một cách "trung gian" giúp hiểu: với mỗi độ dài `L`, tổng số nguyên âm trên mọi cửa sổ độ dài `L` tính được bằng cửa sổ trượt `O(n)`; cộng qua mọi `L` cho `O(n²)`. Không nhanh hơn vét cạn về bậc, nhưng là cầu nối tới các biến thể "chỉ độ dài `k`" (Bài tập 5) và minh họa rằng đóng góp của vị trí `i` cho độ dài `L` là `min(i, n−L, L−1, n−1−i) + 1`.

In [13]:
def vowels_by_length(word):
    n = len(word); total = 0
    for L in range(1, n + 1):
        cur = sum(ch in VOWELS for ch in word[:L]); total += cur
        for r in range(L, n):
            cur += (word[r] in VOWELS) - (word[r - L] in VOWELS); total += cur
    return total

def windows_containing(i, n, L):
    return max(0, min(i, n - L) - max(0, i - L + 1) + 1)

print([vowels_by_length(w) for w, _ in examples])
n = 7
for L in (1, 3, 7):
    print(f"L={L}: số cửa sổ chứa vị trí i =", [windows_containing(i, n, L) for i in range(n)])
assert all(sum(windows_containing(i, n, L) for L in range(1, n + 1)) == (i + 1) * (n - i) for n in range(1, 12) for i in range(n))

[6, 3, 0]
L=1: số cửa sổ chứa vị trí i = [1, 1, 1, 1, 1, 1, 1]
L=3: số cửa sổ chứa vị trí i = [1, 2, 3, 3, 3, 2, 1]
L=7: số cửa sổ chứa vị trí i = [1, 1, 1, 1, 1, 1, 1]


## 8. TRACE từng bước

Trace Cách 3 trên `"aba"`:

In [14]:
def trace(word):
    f = total = 0
    for i, ch in enumerate(word):
        add = i + 1 if ch in VOWELS else 0
        f += add; total += f
        print(f"i={i} '{ch}': f += {add} -> f = {f} (tổng trên các chuỗi con kết thúc ở {i}), total = {total}")
    return total

assert trace("aba") == 6

i=0 'a': f += 1 -> f = 1 (tổng trên các chuỗi con kết thúc ở 0), total = 1
i=1 'b': f += 0 -> f = 1 (tổng trên các chuỗi con kết thúc ở 1), total = 2
i=2 'a': f += 3 -> f = 4 (tổng trên các chuỗi con kết thúc ở 2), total = 6


| i | ký tự | chuỗi con kết thúc ở i | số nguyên âm mỗi chuỗi | f | total |
|---|---|---|---|---|---|
| 0 | a | `a` | 1 | 1 | 1 |
| 1 | b | `ab`, `b` | 1, 0 | 1 | 2 |
| 2 | a | `aba`, `ba`, `a` | 2, 1, 1 | 4 | **6** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — tổng đóng góp `(i + 1)(n − i)`.

In [15]:
class Solution:
    def countVowels(self, word: str) -> int:
        n = len(word)
        return sum((i + 1) * (n - i) for i, ch in enumerate(word) if ch in "aeiou")

for w, exp in examples:
    got = Solution().countVowels(w); print(got, exp); assert got == exp

6 6
3 3
0 0


## 10. Vì sao thuật toán đúng

Gọi `v(S)` là số nguyên âm của chuỗi con `S = word[l..r]`; `v(S) = Σ_{i=l}^{r} [word[i] ∈ V]`. Khi đó

`Σ_S v(S) = Σ_{l ≤ r} Σ_{i=l}^{r} [word[i] ∈ V] = Σ_i [word[i] ∈ V] · #{(l, r) : l ≤ i ≤ r} = Σ_i [word[i] ∈ V] · (i + 1)(n − i)`.

Đổi thứ tự hai tổng hữu hạn là hợp lệ; số cặp `(l, r)` với `0 ≤ l ≤ i ≤ r ≤ n − 1` là `(i + 1)(n − i)`. ∎

**Cách 3** tương đương: `f[i] = Σ_{l ≤ i} v(word[l..i])`; `v(word[l..i]) = v(word[l..i−1]) + [word[i] ∈ V]` cho `l < i` và `[word[i] ∈ V]` cho `l = i` ⇒ `f[i] = f[i−1] + (i + 1)[word[i] ∈ V]`.

### 10.1 Kiểm chứng thực nghiệm

So bốn cách với định nghĩa `O(n³)` trên chuỗi ngắn, và kiểm tra công thức "toàn nguyên âm" `n(n+1)(n+2)/6`.

In [16]:
def by_definition(word):
    n = len(word)
    return sum(sum(ch in VOWELS for ch in word[l:r + 1]) for l in range(n) for r in range(l, n))

rng = random.Random(1010)
for _ in range(500):
    w = "".join(rng.choice("abeixyz") for _ in range(rng.randint(1, 20)))
    ref = by_definition(w)
    assert ref == vowels_bruteforce(w) == vowels_dp(w) == vowels_prefix(w) == Solution().countVowels(w)
for n in range(1, 60):
    assert Solution().countVowels("a" * n) == n * (n + 1) * (n + 2) // 6
print("Định nghĩa O(n^3) = 4 cách (500 chuỗi); công thức toàn nguyên âm đúng")

Định nghĩa O(n^3) = 4 cách (500 chuỗi); công thức toàn nguyên âm đúng


### 10.2 Kiểm chứng: `f[i]` của DP bằng tổng đóng góp "cắt ở `i`"

`f[i]` (tổng trên chuỗi con kết thúc tại `i`) phải bằng `Σ_{j ≤ i, word[j] nguyên âm} (j + 1)` — mỗi nguyên âm ở `j` nằm trong `j + 1` chuỗi con kết thúc tại `i` (chọn đầu `0..j`). Cộng `f[i]` qua `i` cho lại công thức `(j + 1)(n − j)`. Kiểm tra từng `f[i]` trên dữ liệu ngẫu nhiên:

In [17]:
rng = random.Random(102)
for _ in range(500):
    w = "".join(rng.choice("abeio") for _ in range(rng.randint(1, 25))); n = len(w); f = 0
    for i, ch in enumerate(w):
        if ch in VOWELS: f += i + 1
        assert f == sum(j + 1 for j in range(i + 1) if w[j] in VOWELS)
        assert f == sum(sum(c in VOWELS for c in w[l:i + 1]) for l in range(i + 1))
print("f[i] = Σ (j+1) = tổng trên chuỗi con kết thúc tại i — đúng cho mọi i (500 chuỗi)")

f[i] = Σ (j+1) = tổng trên chuỗi con kết thúc tại i — đúng cho mọi i (500 chuỗi)


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi con (cộng dồn) | `O(n²)` | `O(1)` |
| 2. Đóng góp | `O(n)` | `O(1)` |
| 3. DP kết thúc tại `i` | `O(n)` | `O(1)` |
| 4. Tiền tố có trọng số | `O(n)` | `O(n)` |

In [18]:
rng = random.Random(2063)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10 ** 5))
for name, f in [("contrib", Solution().countVowels), ("dp", vowels_dp), ("prefix", vowels_prefix)]:
    t = time.perf_counter(); r = f(big); print(f"{name:<8}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
t = time.perf_counter(); vowels_bruteforce(big[:3000]); print(f"brute n = 3000: {(time.perf_counter() - t) * 1000:.0f} ms")

contrib : 32255957411336 (4 ms)
dp      : 32255957411336 (5 ms)
prefix  : 32255957411336 (13 ms)


brute n = 3000: 219 ms


## 12. Bộ test

In [19]:
ALL = {"contrib": Solution().countVowels, "dp": vowels_dp, "prefix": vowels_prefix}
def run_all(w, brute=True):
    res = {k: f(w) for k, f in ALL.items()}
    if brute:
        res["brute"] = vowels_bruteforce(w)
    assert len(set(res.values())) == 1, (w, res)
    return res["contrib"]

for w, exp in examples:
    assert run_all(w) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [("a", 1), ("b", 0), ("aa", 4), ("aeiou", 35), ("y", 0), ("ba", 2), ("noosabasboosa", None)]
for w, exp in edge:
    got = run_all(w)
    assert exp is None or got == exp, (w, got)
rng = random.Random(20630)
for _ in range(400):
    run_all("".join(rng.choice(rng.choice(["ab", "aeiou", string.ascii_lowercase])) for _ in range(rng.randint(1, 60))))
assert run_all("a" * 10 ** 5, brute=False) == 10 ** 5 * (10 ** 5 + 1) * (10 ** 5 + 2) // 6
print("Ví dụ, 7 edge case, 400 chuỗi ngẫu nhiên, chuỗi 1e5 toàn 'a': OK")

Ví dụ, 7 edge case, 400 chuỗi ngẫu nhiên, chuỗi 1e5 toàn 'a': OK


### 12.1 Kiểm tra tính chất

- Đảo ngược chuỗi không đổi đáp án (đóng góp `(i+1)(n−i)` đối xứng).
- Đổi một phụ âm thành nguyên âm ở vị trí `i` làm đáp án tăng đúng `(i + 1)(n − i)`.
- Nối hai chuỗi: `f(A + B) ≥ f(A) + f(B)`.

In [20]:
rng = random.Random(121)
f = Solution().countVowels
for _ in range(1000):
    w = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 30))); n = len(w); a = f(w)
    assert f(w[::-1]) == a
    cons = [i for i, ch in enumerate(w) if ch not in VOWELS]
    if cons:
        i = rng.choice(cons); w2 = w[:i] + "o" + w[i + 1:]; assert f(w2) - a == (i + 1) * (n - i)
    v = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 10))); assert f(w + v) >= a + f(v)
print("3 tính chất đúng trên 1000 chuỗi")

3 tính chất đúng trên 1000 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số chuỗi con **chứa** nguyên âm (thay vì tổng số nguyên âm)

In [21]:
def wrong_contains(word):
    n = len(word)
    return sum(1 for l in range(n) for r in range(l, n) if any(ch in VOWELS for ch in word[l:r + 1]))   # SAI

for w, exp in examples:
    print(f"{w}: sai {wrong_contains(w)} | đúng {exp}")

aba: sai 5 | đúng 6
abc: sai 3 | đúng 3
ltcd: sai 0 | đúng 0


**Vì sao sai:** chuỗi con `"aba"` có **hai** nguyên âm, phải đếm 2 chứ không phải 1. **Sửa:** cộng số nguyên âm.

### Lỗi 2 — Lệch một: `i · (n − i)` hoặc `(i + 1)(n − i − 1)`

In [22]:
def wrong_offset(word):
    n = len(word)
    return sum(i * (n - i) for i, ch in enumerate(word) if ch in VOWELS)                 # SAI

for w, exp in examples:
    print(f"{w}: sai {wrong_offset(w)} | đúng {exp}")

aba: sai 2 | đúng 6
abc: sai 0 | đúng 3
ltcd: sai 0 | đúng 0


**Vì sao sai:** có `i + 1` lựa chọn điểm đầu (`0..i`) — chỉ số 0-based. **Sửa:** kiểm tra công thức với chuỗi dài 1: `(0 + 1)(1 − 0) = 1`.

### Lỗi 3 — Tràn số 32-bit (ở ngôn ngữ khác)

In [23]:
def as_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= 1 << 31 else x

w = "a" * 3000                                   # tự dựng (đối chiếu lại đề gốc)
print("đúng:", Solution().countVowels(w), "| nếu cộng dồn bằng int32:", as_int32(Solution().countVowels(w)))

đúng: 4504501000 | nếu cộng dồn bằng int32: 209533704


**Vì sao sai:** đề cảnh báo đáp án có thể vượt `int32`; với `n = 3000` toàn nguyên âm đã là `4.5·10⁹`. Python không lỗi, nhưng C++/Java cần `long long`/`long` — kể cả trong phép nhân `(i + 1)·(n − i)`.

### Lỗi 4 — Coi `'y'` (hoặc chữ hoa) là nguyên âm

In [24]:
def wrong_y(word):
    n = len(word)
    return sum((i + 1) * (n - i) for i, ch in enumerate(word) if ch in "aeiouy")          # SAI

for w in ("y", "byte", "ltcd"):
    print(f"{w}: sai {wrong_y(w)} | đúng {Solution().countVowels(w)}")

y: sai 1 | đúng 0
byte: sai 10 | đúng 4
ltcd: sai 0 | đúng 0


**Vì sao sai:** đề liệt kê đúng năm nguyên âm `a, e, i, o, u`. **Sửa:** dùng đúng tập.

### Lỗi 5 — DP quên cộng `f` vào tổng ở mỗi bước

In [25]:
def wrong_dp(word):
    f = 0
    for i, ch in enumerate(word):
        if ch in VOWELS: f += i + 1
    return f                                                                               # SAI: chỉ là f[n-1]

for w, exp in examples:
    print(f"{w}: sai {wrong_dp(w)} | đúng {exp}")

aba: sai 4 | đúng 6
abc: sai 1 | đúng 3
ltcd: sai 0 | đúng 0


**Vì sao sai:** `f` chỉ là tổng trên chuỗi con **kết thúc tại vị trí cuối**; đáp án là tổng `f[i]` qua **mọi** `i`. **Sửa:** `total += f` mỗi bước.

### Lỗi 6 — Liệt kê chuỗi con bằng cắt chuỗi (`O(n³)`)

In [26]:
for n in (100, 200, 400):
    w = "".join(random.Random(n).choice("ab") for _ in range(n))
    t = time.perf_counter(); by_definition(w); t1 = time.perf_counter() - t
    t = time.perf_counter(); Solution().countVowels(w); t2 = time.perf_counter() - t
    print(f"n={n}: cắt chuỗi O(n^3) {t1 * 1000:6.0f} ms | đóng góp {t2 * 1000:.2f} ms")

n=100: cắt chuỗi O(n^3)      5 ms | đóng góp 0.01 ms
n=200: cắt chuỗi O(n^3)     34 ms | đóng góp 0.02 ms


n=400: cắt chuỗi O(n^3)    254 ms | đóng góp 0.02 ms


**Vì sao không tốt:** tăng gấp 8 khi `n` gấp đôi. **Sửa:** đóng góp `O(n)`.

### Lỗi 7 — Đếm trên tập chuỗi con **khác nhau** (dùng `set`)

In [27]:
def wrong_distinct(word):
    n = len(word)
    subs = {word[l:r + 1] for l in range(n) for r in range(l, n)}           # SAI: gộp chuỗi con trùng nội dung
    return sum(sum(ch in VOWELS for ch in t) for t in subs)

for w, exp in examples:
    print(f"{w}: sai {wrong_distinct(w)} | đúng {exp}")

aba: sai 5 | đúng 6
abc: sai 3 | đúng 3
ltcd: sai 0 | đúng 0


**Vì sao sai:** chuỗi con được phân biệt theo **vị trí**: `"aba"` có hai chuỗi con `"a"` (vị trí 0 và 2), cả hai đều được tính. Dùng `set` gộp chúng làm mất 1. **Sửa:** đếm theo cặp chỉ số `(l, r)` — hoặc đơn giản dùng công thức đóng góp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng một đại lượng **cộng tính** trên mọi chuỗi con/mảng con" ⇒ **đóng góp từng phần tử** × số chuỗi con chứa nó.
- Số mảng con chứa vị trí `i`: `(i + 1)(n − i)`.
- Đại lượng không cộng tính (min, max, số phần tử khác nhau) ⇒ đóng góp với **biên** (stack đơn điệu, vị trí xuất hiện trước).
- Cảnh báo "không vừa 32-bit" ⇒ chú ý kiểu số.

## 15. Biến thể của bài

1. **828 Count Unique Characters of All Substrings**: đóng góp của ký tự với vị trí xuất hiện trước/sau.
2. **907 Sum of Subarray Minimums**: đóng góp với stack đơn điệu.
3. **2262 Total Appeal of A String**: số ký tự **khác nhau** trên mọi chuỗi con — đóng góp `(i − last[c])·(n − i)`.
4. **1588 Sum of All Odd Length Subarrays**: đếm số mảng con độ dài lẻ chứa `i`.
5. **Tổng số nguyên âm trên chuỗi con độ dài đúng `k`** — cửa sổ trượt.

### 15.1 Code biến thể: 2262 Total Appeal (đóng góp với lần xuất hiện trước)

In [28]:
def appeal_sum(s):
    last = {}; n = len(s); total = 0
    for i, ch in enumerate(s):
        total += (i - last.get(ch, -1)) * (n - i)          # ch là "lần xuất hiện đầu" trong các chuỗi con này
        last[ch] = i
    return total

def appeal_brute(s):
    n = len(s); return sum(len(set(s[l:r + 1])) for l in range(n) for r in range(l, n))

rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15))); assert appeal_sum(s) == appeal_brute(s)
print("2262 khớp vét cạn;", appeal_sum("abbca"), "(LeetCode: 28)")

2262 khớp vét cạn; 28 (LeetCode: 28)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cộng dồn mọi chuỗi con | Vét cạn | `O(n²)` | `O(1)` | Kiểm chứng |
| 2. Đóng góp | `(i+1)(n−i)` | `O(n)` | `O(1)` | **Bản nộp** |
| 3. DP kết thúc tại `i` | `f += i + 1` | `O(n)` | `O(1)` | Tổng quát hóa |
| 4. Tiền tố có trọng số | `Σ P[j](2j − n)` | `O(n)` | `O(n)` | Góc nhìn đại số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"a"`; (b) `"aa"`; (c) `"aeiou"`; (d) `"ba"`.

**Bài 2 (tính tay):** Bảng đóng góp cho `"leet"`.

**Bài 3 (điền TODO):** Hoàn thiện `contribution(i, n)`.

**Bài 4 (tự viết lại):** Viết `vowels_odd_length(word)` — tổng số nguyên âm chỉ trên các chuỗi con **độ dài lẻ**, `O(n)`; đối chiếu vét cạn.

**Bài 5 (mini challenge):** Viết `vowels_at_most_k(word, k)` — tổng số nguyên âm trên các chuỗi con có độ dài **≤ k**, `O(n)` (đếm số chuỗi con độ dài `≤ k` chứa `i` bằng công thức); đối chiếu vét cạn.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def contribution(i, n):
    # TODO: số chuỗi con của chuỗi dài n chứa vị trí i
    pass

In [30]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def vowels_odd_length(word):
    # TODO: đếm cặp (l, r) với l <= i <= r và (r - l) chẵn: tách theo tính chẵn lẻ của l và r
    pass

def vowels_at_most_k(word, k):
    # TODO: với mỗi l trong [max(0, i-k+1), i]: số r hợp lệ = min(n-1, l+k-1) - i + 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: công thức đóng góp. Mức 2: cộng các nguyên âm. Mức 3: (a) `1`; (b) `2 + 2 = 4`; (c) `5 + 8 + 9 + 8 + 5 = 35`; (d) `2` (chỉ `a` ở vị trí 1: `2·1`).

**Bài 2.** Mức 1: `n = 4`, nguyên âm ở 1, 2. Mức 2: `2·3 = 6` và `3·2 = 6`. Mức 3: `12`.

**Bài 3.** Mức 1: số cách chọn đầu. Mức 2: số cách chọn cuối. Mức 3: nhân lại.

**Bài 4.** Mức 1: độ dài lẻ ⇔ `l` và `r` cùng tính chẵn lẻ. Mức 2: đếm `l ∈ [0, i]` chẵn/lẻ và `r ∈ [i, n−1]` chẵn/lẻ. Mức 3: `chẵn_l·chẵn_r + lẻ_l·lẻ_r`.

**Bài 5.** Mức 1: cố định `l`, `r ∈ [i, min(n−1, l+k−1)]`. Mức 2: tổng qua `l` là tổng một dãy có phần "bị chặn" bởi `n − 1` — tính bằng công thức cấp số cộng hoặc vòng lặp `O(k)` cho mỗi `i` (chấp nhận `O(nk)` ban đầu). Mức 3: tối ưu bằng công thức đóng.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1, 2, 3
f = Solution().countVowels
b1 = [f(w) for w in ("a", "aa", "aeiou", "ba")]
print("Bài 1:", b1); assert b1 == [1, 4, 35, 2]
print("Bài 2:", [(i, ch, (i + 1) * (4 - i)) for i, ch in enumerate("leet") if ch in VOWELS], f("leet")); assert f("leet") == 12
def contribution(i, n):
    return (i + 1) * (n - i)
assert all(contribution(i, n) == sum(1 for l in range(n) for r in range(l, n) if l <= i <= r) for n in range(1, 9) for i in range(n))
print("Bài 3: OK")

Bài 1: [1, 4, 35, 2]
Bài 2: [(1, 'e', 6), (2, 'e', 6)] 12
Bài 3: OK


In [32]:
# Bài 4, 5
def count_parity(lo, hi):
    # số số chẵn, lẻ trong [lo, hi]
    if lo > hi: return 0, 0
    total = hi - lo + 1; even = hi // 2 - (lo - 1) // 2 if lo > 0 else hi // 2 + 1
    return even, total - even

def vowels_odd_length(word):
    n = len(word); s = 0
    for i, ch in enumerate(word):
        if ch in VOWELS:
            le, lo_ = count_parity(0, i); re, ro = count_parity(i, n - 1)
            s += le * re + lo_ * ro
    return s

def at_most_k_count(i, n, k):
    total = 0
    for l in range(max(0, i - k + 1), i + 1):
        total += min(n - 1, l + k - 1) - i + 1
    return total

def vowels_at_most_k(word, k):
    n = len(word)
    return sum(at_most_k_count(i, n, k) for i, ch in enumerate(word) if ch in VOWELS)

rng = random.Random(5)
for _ in range(300):
    w = "".join(rng.choice("abe") for _ in range(rng.randint(1, 14))); n = len(w); k = rng.randint(1, n)
    assert vowels_odd_length(w) == sum(sum(ch in VOWELS for ch in w[l:r + 1]) for l in range(n) for r in range(l, n) if (r - l) % 2 == 0)
    assert vowels_at_most_k(w, k) == sum(sum(ch in VOWELS for ch in w[l:r + 1]) for l in range(n) for r in range(l, min(n, l + k)))
print("Bài 4, 5: OK |", vowels_odd_length("aba"), vowels_at_most_k("aba", 2))

Bài 4, 5: OK | 4 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đóng góp):**

```python
n = len(word)
return sum((i + 1) * (n - i) for i, ch in enumerate(word) if ch in "aeiou")
```

**Template (DP):** `f += (i + 1) nếu nguyên âm; total += f`.

**Checklist:**
- [ ] Cộng **số** nguyên âm, không phải đếm chuỗi con có nguyên âm?
- [ ] `(i + 1)(n − i)` với chỉ số 0-based?
- [ ] Đúng năm nguyên âm `aeiou`?
- [ ] Kiểu số 64-bit ở ngôn ngữ khác?
- [ ] `O(n)`?